In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input  # Corrected import for preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import UpSampling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model


In [ ]:
# Load ResNet50 pretrained model as an encoder
resnet_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Define the model up to the last convolutional layer
encoder_model = Model(inputs=resnet_model.input, outputs=resnet_model.output)

# Freeze the initial layers of the encoder and unfreeze the last few layers
for layer in encoder_model.layers[:-20]:  # Freeze all layers except the last 10
    layer.trainable = False

for layer in encoder_model.layers[-20:]:  # Unfreeze the last 10 layers
    layer.trainable = True

In [ ]:

#! pip install split-folders
import splitfolders
 # Split the data into train, val, test sets
import os

#from google.colab import drive
#drive.mount('/content/drive')


# Define paths
input_path = '/content/drive/MyDrive/Test-New/Test'  # Contains 'fake' and 'real' folders
output_path = '/content/drive/MyDrive/Test-New-Split'  # Folder to store the split data




In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create an ImageDataGenerator for loading and preprocessing the images
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Apply to your generators
train_generator = datagen.flow_from_directory(
    os.path.join(output_path, 'train'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    os.path.join(output_path, 'val'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

test_generator = datagen.flow_from_directory(
    os.path.join(output_path, 'test'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)
print("Class indices for training data:", train_generator.class_indices)
print("Class indices for validation data:", val_generator.class_indices)
print("Class indices for test data:", test_generator.class_indices)

In [ ]:
# Count the number of real and fake images in the training set
train_class_counts = dict(zip(*np.unique(train_generator.classes, return_counts=True)))
print("Training set class distribution:", train_class_counts)

# Count the number of real and fake images in the validation set
val_class_counts = dict(zip(*np.unique(val_generator.classes, return_counts=True)))
print("Validation set class distribution:", val_class_counts)

# Count the number of real and fake images in the test set
test_class_counts = dict(zip(*np.unique(test_generator.classes, return_counts=True)))
print("Test set class distribution:", test_class_counts)


In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model

# Define paths
train_path = 'data/Test'
val_path = 'data/Val'
test_path = 'data/Test'
encoded_path = 'data'

# Function to check directory contents
def check_directory(directory):
    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
    else:
        print(f"Directory contents for {directory}:")
        for subdir in ['Real', 'Fake']:
            sub_path = os.path.join(directory, subdir)
            if os.path.exists(sub_path):
                print(f"  {subdir} contains {len(os.listdir(sub_path))} files.")
            else:
                print(f"  Subdirectory '{subdir}' does not exist within {directory}.")

# Check directories
check_directory(train_path)
check_directory(val_path)
check_directory(test_path)

# Function to encode images and map each filename to its encoded representation
def encode_images_with_filenames(directory, encoder_model, datagen, save_directory):
    generator = datagen.flow_from_directory(
        directory,
        target_size=(224, 224),
        batch_size=16,  # Reduced batch size to reduce memory usage
        class_mode=None,  # Only need the images, no labels
        shuffle=False
    )

    # Process each batch of images
    for i, batch in enumerate(generator):
        # Encode the batch of images
        encoded_batch = encoder_model.predict(batch)

        # Get the filenames for the current batch
        filenames = generator.filenames[i * generator.batch_size:(i + 1) * generator.batch_size]

        # Save each encoded feature immediately to avoid high memory usage
        for j, filename in enumerate(filenames):
            # Determine the subdirectory (Real or Fake) based on the directory structure
            if 'Real' in filename:
                subdir = 'Real'
            else:
                subdir = 'Fake'

            # Create the path to save the encoded features
            file_path = os.path.join(save_directory, subdir, filename)
            # Create the subdirectory if it doesn't exist
            os.makedirs(os.path.dirname(file_path), exist_ok=True)
            # Save the encoded features as a .npy file
            np.save(file_path.replace('.jpg', '.npy'), encoded_batch[j])

        # Clear memory after processing each batch
        del batch, encoded_batch
        import gc; gc.collect()  # Run garbage collection

        # Stop the loop after all images are processed
        if (i + 1) * generator.batch_size >= generator.samples:
            break

# Prepare the encoder and data generator
datagen = ImageDataGenerator(rescale=1./255)
encoder_model = Model(inputs=resnet_model.input, outputs=resnet_model.output)  # Ensure your encoder model is defined here

# Create directories for saving encoded files
for subset in ['train', 'val', 'test']:
    for subfolder in ['Real', 'Fake']:
        os.makedirs(os.path.join(encoded_path, subset, subfolder), exist_ok=True)

# Encode and save features for each subset of the data
encode_images_with_filenames(train_path, encoder_model, datagen, os.path.join(encoded_path, 'train'))
encode_images_with_filenames(val_path, encoder_model, datagen, os.path.join(encoded_path, 'val'))
encode_images_with_filenames(test_path, encoder_model, datagen, os.path.join(encoded_path, 'test'))

# Print summary
print("Encoding and saving complete.")


In [ ]:
import os
import numpy as np

# Define the path where encoded files are stored
encoded_path = '/content/drive/MyDrive/Test-New-Split/Imagenet'

# Function to load and display first 5 encoded features
def load_and_display_encoded_features(base_path, subset):
    print(f"--- Displaying first 5 encoded features for {subset} ---")

    for label in ['Real', 'Fake']:
        # Full path for the subset and label (including sub-subdirectory if present)
        full_path = os.path.join(base_path, subset, label)

        # Traverse through subdirectories and collect .npy files
        npy_files = []
        for root, dirs, files in os.walk(full_path):
            for file in files:
                if file.endswith('.npy'):
                    npy_files.append(os.path.join(root, file))

        print(f"{label} Images Encoded Features:")
        if npy_files:
            # Load and display the first 5 files
            for i, npy_file in enumerate(npy_files[:5]):
                encoded_feature = np.load(npy_file)
                print(f"{os.path.basename(npy_file)} - Encoded Feature: {encoded_feature[:5]}...")  # Display first 5 elements
        else:
            print(f"No .npy files found in {label} for {subset}.")

# Check and display for each subset
for subset in ['train', 'val', 'test']:
    load_and_display_encoded_features(encoded_path, subset)


In [ ]:
def check_npy_files(directory):
    for label_name in ['Real', 'Fake']:
        label_dir = os.path.join(directory, label_name)
        if os.path.exists(label_dir):
            # Use os.walk to find .npy files recursively
            npy_files = []
            for subdir, _, files in os.walk(label_dir):
                npy_files.extend([f for f in files if f.endswith('.npy')])
            print(f"{label_name} contains {len(npy_files)} files.")
        else:
            print(f"{label_name} directory does not exist.")

# Check the train, validation, and test directories
check_npy_files(os.path.join(encoded_path, 'train'))
check_npy_files(os.path.join(encoded_path, 'val'))
check_npy_files(os.path.join(encoded_path, 'test'))


In [ ]:
import os
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, Activation, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.utils import shuffle
from tensorflow.keras.utils import Sequence

# Paths for the encoded features
train_path = '/content/drive/MyDrive/Test-New-Split/Imagenet/train'
val_path = '/content/drive/MyDrive/Test-New-Split/Imagenet/val'
test_path = '/content/drive/MyDrive/Test-New-Split/Imagenet/test'

# Function to load encoded features and labels from nested directories
def load_encoded_features(encoded_dir):
    features, labels = [], []
    for label_name in ['Real', 'Fake']:
        label = 1 if label_name == 'Real' else 0
        label_dir = os.path.join(encoded_dir, label_name)
        for subdir, _, files in os.walk(label_dir):
            for file in files:
                if file.endswith('.npy'):
                    feature = np.load(os.path.join(subdir, file))
                    features.append(feature)
                    labels.append(label)
    return shuffle(np.array(features), np.array(labels))

# Load the data
encoded_train, train_labels = load_encoded_features(train_path)
encoded_val, val_labels = load_encoded_features(val_path)
encoded_test, test_labels = load_encoded_features(test_path)

print("Shapes of datasets:")
print("Train:", encoded_train.shape, train_labels.shape)
print("Validation:", encoded_val.shape, val_labels.shape)
print("Test:", encoded_test.shape, test_labels.shape)

# Flatten the features to match the classifier input
X_train_flat = encoded_train.reshape(encoded_train.shape[0], -1)
X_val_flat = encoded_val.reshape(encoded_val.shape[0], -1)
X_test_flat = encoded_test.reshape(encoded_test.shape[0], -1)

print("Shapes after flattening:")
print("X_train_flat:", X_train_flat.shape)
print("X_val_flat:", X_val_flat.shape)
print("X_test_flat:", X_test_flat.shape)

# Define a data generator
class DataGenerator(Sequence):
    def __init__(self, features, labels, batch_size=32):
        self.features = features
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.features))

    def __len__(self):
        return int(np.ceil(len(self.features) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch_features = self.features[start:end]
        batch_labels = self.labels[start:end]
        return batch_features, batch_labels

# Learning rate scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)



# Define the classifier model
classifier_model = Sequential()
classifier_model.add(Input(shape=(X_train_flat.shape[1],)))  # Define input shape using Input layer

classifier_model.add(Dense(512, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(512, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(256, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(256, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(128, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(128, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(Dropout(0.3))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(64, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(Dropout(0.3))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(32, kernel_regularizer=l2(0.0001)))
classifier_model.add(Activation('softplus'))
classifier_model.add(Dropout(0.3))
classifier_model.add(BatchNormalization())

classifier_model.add(Dense(1, activation='sigmoid'))

# Compile the model
optimizer = RMSprop(learning_rate=0.00001)
classifier_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Create data generators
train_generator = DataGenerator(X_train_flat, train_labels, batch_size=32)
val_generator = DataGenerator(X_val_flat, val_labels, batch_size=32)

# Train the classifier model using the generator
history = classifier_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    #class_weight=class_weight,  # Uncomment this line if class weighting is needed
    callbacks=[lr_scheduler]
)

# Evaluate on the test set
test_loss, test_accuracy = classifier_model.evaluate(X_test_flat, test_labels)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
